In [ ]:
import requests
import urllib.request
import rasterio
from rasterio.enums import Resampling as RioResampling
from rasterio.mask import mask
import geopandas as gpd
import numpy as np
import xml.etree.ElementTree as ET
from pathlib import Path

# --- Setup ---
cdl_dir = Path("../../CDL")
cdl_dir.mkdir(parents=True, exist_ok=True)

counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip")
iowa = counties[counties["STATEFP"] == "19"].to_crs("EPSG:5070")
iowa_geom = [iowa.union_all()]

def get_iowa_cdl_url(year):
    url = f"https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLFile?year={year}&fips=19"
    r = requests.get(url, timeout=30)
    root = ET.fromstring(r.text)
    for elem in root.iter():
        if elem.tag.endswith("returnURL"):
            return elem.text
    raise ValueError(f"No returnURL in response for {year}: {r.text}")

years = range(2000, 2026)  # CDL available up to ~2025 for Iowa

for year in years:
    out_path = cdl_dir / f"CDL_{year}_19_1km.tif"

    if out_path.exists():
        print(f"{year}: already exists, skipping")
        continue

    # --- Step 1: get download URL ---
    try:
        file_url = get_iowa_cdl_url(year)
        print(f"{year}: downloading from {file_url}")
    except Exception as e:
        print(f"{year}: failed to get URL - {e}")
        continue

    # --- Step 2: download raw CDL to temp file ---
    raw_path = cdl_dir / f"CDL_{year}_19_raw.tif"
    try:
        urllib.request.urlretrieve(file_url, raw_path)
    except Exception as e:
        print(f"{year}: download failed - {e}")
        continue

    # --- Step 3: resample to 1km (mode), clip to Iowa, save ---
    try:
        with rasterio.open(raw_path) as src:
            print(f"{year}: raw shape={src.shape}, res={src.res}, crs={src.crs}")

            # resample to 1km using mode (most common crop class per cell)
            native_res = src.res[0]  # e.g. 30m
            scale = native_res / 1000
            new_h = int(src.height * scale)
            new_w = int(src.width  * scale)
            new_transform = src.transform * src.transform.scale(src.width/new_w, src.height/new_h)

            cdl_1km = src.read(
                1,
                out_shape=(new_h, new_w),
                resampling=RioResampling.mode
            )

            profile = src.profile.copy()
            profile.update(
                height=new_h, width=new_w,
                transform=new_transform,
                crs="EPSG:5070",
                nodata=0
            )

            # clip to Iowa boundary
            with rasterio.io.MemoryFile() as memfile:
                with memfile.open(**profile) as tmp:
                    tmp.write(cdl_1km, 1)
                with memfile.open() as tmp:
                    cdl_clip, clip_transform = mask(tmp, iowa_geom, crop=True, nodata=0)
                    cdl_clip = cdl_clip[0]
                    clip_profile = tmp.profile.copy()
                    clip_profile.update(
                        height=cdl_clip.shape[0],
                        width=cdl_clip.shape[1],
                        transform=clip_transform
                    )

        # save clipped 1km raster
        with rasterio.open(out_path, "w", **clip_profile) as dst:
            dst.write(cdl_clip, 1)

        print(f"{year}: saved {out_path.name}, shape={cdl_clip.shape}, valid pixels={(cdl_clip > 0).sum()}")

    except Exception as e:
        print(f"{year}: processing failed - {e}")

    finally:
        # clean up raw temp file
        if raw_path.exists():
            raw_path.unlink()

print("Done.")

In [ ]:
from pyproj import Transformer


cdl_dir = Path("../../CDL")

cdl_legend = {
    1:"Corn", 5:"Soybeans", 36:"Alfalfa", 37:"Other Hay",
    61:"Fallow/Idle", 111:"Open Water", 121:"Developed",
    141:"Deciduous Forest", 176:"Grass/Pasture", 190:"Woody Wetlands"
}

years = sorted([int(p.stem.split("_")[1]) for p in cdl_dir.glob("CDL_*_19_1km.tif")])

# --- File 1: build cell grid from reference year ---
ref_path = cdl_dir / f"CDL_{years[0]}_19_1km.tif"

with rasterio.open(ref_path) as src:
    data = src.read(1)
    transform = src.transform
    crs = src.crs

# transform from EPSG:5070 to WGS84 lat/lon
transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)

valid_rows, valid_cols = np.where(data > 0)

# compute cell center coordinates in EPSG:5070 then convert to lat/lon
xs, ys = rasterio.transform.xy(transform, valid_rows, valid_cols, offset="center")
lons, lats = transformer.transform(xs, ys)

cell_grid = pd.DataFrame({
    "cell_id": np.arange(len(valid_rows)),
    "row":     valid_rows,
    "col":     valid_cols,
    "longitude": np.round(lons, 6),
    "latitude":  np.round(lats, 6),
})
cell_grid.to_csv("iowa_cell_grid.csv", index=False)
print(f"Cell grid: {len(cell_grid)} cells")

# build row/col -> cell_id lookup for fast annual mapping
rc_to_id = {(r, c): i for i, r, c in zip(cell_grid["cell_id"], cell_grid["row"], cell_grid["col"])}

# --- File 2: annual crop per cell ---
all_years = []

for year in years:
    path = cdl_dir / f"CDL_{year}_19_1km.tif"
    with rasterio.open(path) as src:
        data = src.read(1)

    rows, cols = np.where(data > 0)
    values = data[rows, cols]

    df = pd.DataFrame({
        "cell_id":   [rc_to_id.get((r, c), -1) for r, c in zip(rows, cols)],
        "year":      year,
        "cdl_class": values.astype(int),
    })
    df["crop"] = df["cdl_class"].map(cdl_legend).fillna("Other")
    df = df[df["cell_id"] >= 0]  # drop any cells not in reference grid
    all_years.append(df)
    print(f"{year}: {len(df)} cells")

annual = pd.concat(all_years, ignore_index=True)
annual.to_csv("iowa_cdl_annual.csv", index=False)
print(f"Annual crop data: {annual.shape}")

In [ ]:
with rasterio.open("../../CDL/CDL_2017_19_1km.tif") as src:
    print(src.meta)
    print("Unique classes:", np.unique(src.read(1))[:20])

In [ ]:
import pandas as pd

cdl_dir = Path("../../CDL")

cdl_legend = {
    1: "Corn", 2: "Cotton", 3: "Rice", 4: "Sorghum", 5: "Soybeans",
    6: "Sunflower", 10: "Peanuts", 11: "Tobacco", 12: "Sweet Corn",
    13: "Pop or Orn Corn", 14: "Mint", 21: "Winter Wheat", 22: "Durum Wheat",
    23: "Spring Wheat", 24: "Winter Wheat", 25: "Other Small Grains",
    26: "Winter Wheat/Soybeans", 27: "Rye", 28: "Oats", 29: "Millet",
    30: "Speltz", 31: "Canola", 32: "Flaxseed", 33: "Safflower",
    34: "Rape Seed", 35: "Mustard", 36: "Alfalfa", 37: "Other Hay/Non Alfalfa",
    38: "Camelina", 39: "Buckwheat", 41: "Sugarbeets", 42: "Dry Beans",
    43: "Potatoes", 44: "Other Crops", 45: "Sugarcane", 46: "Sweet Potatoes",
    47: "Misc Vegs & Fruits", 48: "Watermelons", 49: "Onions", 50: "Cucumbers",
    51: "Chick Peas", 52: "Lentils", 53: "Peas", 54: "Tomatoes",
    55: "Caneberries", 56: "Hops", 57: "Herbs", 58: "Clover/Wildflowers",
    59: "Sod/Grass Seed", 60: "Switchgrass", 61: "Fallow/Idle Cropland",
    63: "Forest", 64: "Shrubland", 65: "Barren", 66: "Cherries",
    67: "Peaches", 68: "Apples", 69: "Grapes", 70: "Christmas Trees",
    71: "Other Tree Crops", 72: "Citrus", 74: "Pecans", 75: "Almonds",
    76: "Walnuts", 77: "Pears", 81: "Clouds/No Data", 82: "Developed",
    83: "Water", 87: "Wetlands", 88: "Nonag/Undefined",
    92: "Aquaculture", 111: "Open Water", 112: "Perennial Ice/Snow",
    121: "Developed/Open Space", 122: "Developed/Low Intensity",
    123: "Developed/Med Intensity", 124: "Developed/High Intensity",
    131: "Barren", 141: "Deciduous Forest", 142: "Evergreen Forest",
    143: "Mixed Forest", 152: "Shrubland", 176: "Grass/Pasture",
    190: "Woody Wetlands", 195: "Herbaceous Wetlands",
    204: "Pistachios", 205: "Triticale", 206: "Carrots", 207: "Asparagus",
    208: "Garlic", 209: "Cantaloupes", 210: "Prunes", 211: "Olives",
    212: "Oranges", 213: "Honeydew Melons", 214: "Broccoli",
    216: "Peppers", 217: "Pomegranates", 218: "Nectarines", 219: "Greens",
    220: "Plums", 221: "Strawberries", 222: "Squash", 223: "Apricots",
    224: "Vetch", 225: "Dbl Crop Winter Wheat/Corn", 226: "Dbl Crop Oats/Corn",
    227: "Lettuce", 229: "Pumpkins", 230: "Dbl Crop Lettuce/Durum Wheat",
    231: "Dbl Crop Lettuce/Cantaloupe", 232: "Dbl Crop Lettuce/Cotton",
    233: "Dbl Crop Lettuce/Barley", 234: "Dbl Crop Durum Wheat/Sorghum",
    235: "Dbl Crop Barley/Corn", 236: "Dbl Crop Winter Wheat/Sorghum",
    237: "Dbl Crop Barley/Sorghum", 238: "Dbl Crop Winter Wheat/Cotton",
    239: "Dbl Crop Soybeans/Cotton", 240: "Dbl Crop Soybeans/Oats",
    241: "Dbl Crop Corn/Soybeans", 242: "Blueberries", 243: "Cabbage",
    244: "Cauliflower", 245: "Celery", 246: "Radishes", 247: "Turnips",
    248: "Eggplant", 249: "Gourds", 250: "Cranberries", 254: "Dbl Crop Barley/Soybeans"
}

years = sorted([
    int(p.stem.split("_")[1])
    for p in cdl_dir.glob("CDL_*_19_1km.tif")
])
print(f"Found {len(years)} years: {years}")

# --- Build pixel count table per crop per year ---
records = []
for year in years:
    path = cdl_dir / f"CDL_{year}_19_1km.tif"
    with rasterio.open(path) as src:
        data = src.read(1)
        total_valid = (data > 0).sum()
        unique, counts = np.unique(data[data > 0], return_counts=True)
        for cls, cnt in zip(unique, counts):
            records.append({
                "year": year,
                "cdl_class": int(cls),
                "crop": cdl_legend.get(int(cls), f"Unknown ({cls})"),
                "pixel_count": int(cnt),
                "total_valid": int(total_valid)
            })
    print(f"{year}: {total_valid} valid pixels, {len(unique)} classes")

df = pd.DataFrame(records)
df["fraction"] = df["pixel_count"] / df["total_valid"]
df["pct"] = df["fraction"] * 100
df.to_csv("iowa_cdl_crop_composition_1km.csv", index=False)
print(df.head())

In [ ]:
df["crop"] = df["crop"].replace("Forest", "Deciduous Forest")
top_crops = (df.groupby("crop")["pixel_count"].sum()
               .sort_values(ascending=False)
               .head(10).index.tolist())
print(top_crops)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

df = pd.read_csv("iowa_cdl_crop_composition_1km.csv")

# --- 1. Top crops overall across all years ---
top_crops = (df.groupby("crop")["pixel_count"].sum()
               .sort_values(ascending=False)
               .head(10).index.tolist())
print("Top 10 crops by total pixels:\n", top_crops)

# --- 2. Stacked area chart — composition over time ---
pivot = (df[df["crop"].isin(top_crops)]
           .pivot_table(index="year", columns="crop", values="pct", aggfunc="sum")
           .fillna(0))

fig, ax = plt.subplots(figsize=(14, 6))
pivot.plot.area(ax=ax, colormap="tab10", alpha=0.85)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlabel("Year")
ax.set_ylabel("% of Iowa land area")
ax.set_title("Iowa Crop Composition Over Time (CDL 1km)")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig("iowa_crop_composition_stacked.png", dpi=150)
plt.show()

# --- 3. Line chart — top crops only, easier to read individual trends ---
fig, ax = plt.subplots(figsize=(14, 6))
for crop in top_crops:
    sub = df[df["crop"] == crop].sort_values("year")
    ax.plot(sub["year"], sub["pct"], marker="o", label=crop)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xlabel("Year")
ax.set_ylabel("% of Iowa land area")
ax.set_title("Top Iowa Crops — Land Share Over Time")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig("iowa_crop_trends_lines.png", dpi=150)
plt.show()

# --- 4. Heatmap — year x crop, % coverage ---
pivot_heat = (df[df["crop"].isin(top_crops)]
                .pivot_table(index="crop", columns="year", values="pct", aggfunc="sum")
                .fillna(0))

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(pivot_heat, annot=True, fmt=".1f", cmap="YlOrRd",
            linewidths=0.5, ax=ax, cbar_kws={"label": "% land area"})
ax.set_title("Iowa Crop Coverage Heatmap (% of land area per year)")
plt.tight_layout()
plt.savefig("iowa_crop_heatmap.png", dpi=150)
plt.show()

# --- 5. Single year snapshot — bar chart ---
year_snap = 2017
snap = df[df["year"] == year_snap].sort_values("pct", ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(snap["crop"], snap["pct"], color=sns.color_palette("tab10", len(snap)))
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_xticklabels(snap["crop"], rotation=45, ha="right")
ax.set_title(f"Iowa Crop Composition {year_snap}")
ax.set_ylabel("% of land area")
plt.tight_layout()
plt.savefig(f"iowa_crop_bar_{year_snap}.png", dpi=150)
plt.show()

In [ ]:
import matplotlib.patches as mpatches
counties = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2023/COUNTY/tl_2023_us_county.zip")
iowa = counties[counties["STATEFP"] == "19"].to_crs("EPSG:5070")

crop_colors = {
    1:   ("#FFD700", "Corn"),
    5:   ("#267000", "Soybeans"),
    176: ("#B8D68C", "Grass/Pasture"),
    141: ("#736D2A", "Deciduous Forest"),
    37:  ("#78BE20", "Other Hay"),
    190: ("#7FC7C7", "Woody Wetlands"),
    61:  ("#BEBEBF", "Fallow/Idle"),
    111: ("#4B6EA9", "Open Water"),
    121: ("#D3D3D3", "Developed"),
}

year = 2011
with rasterio.open(f"../../CDL/CDL_{year}_19_1km.tif") as src:
    data = src.read(1)
    print(data.shape)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

# Build RGB image
rgb = np.ones((*data.shape, 3))
for cls, (color, name) in crop_colors.items():
    r, g, b = int(color[1:3],16)/255, int(color[3:5],16)/255, int(color[5:7],16)/255
    mask = data == cls
    rgb[mask] = [r, g, b]

fig, ax = plt.subplots(figsize=(12, 10))
ax.imshow(rgb, extent=extent)
iowa.boundary.plot(ax=ax, color="black", linewidth=0.5)

patches = [mpatches.Patch(color=c, label=n) for cls,(c,n) in crop_colors.items()]
ax.legend(handles=patches, loc="lower left", fontsize=8)
ax.set_title(f"Iowa Dominant Crop {year}")
plt.tight_layout()
plt.savefig(f"iowa_crop_map_{year}.png", dpi=150)
plt.show()

In [ ]:
import json

cdl_dir = Path("../../CDL")
years = sorted([int(p.stem.split("_")[1]) for p in cdl_dir.glob("CDL_*_19_1km.tif")])

cdl_legend = {
    1: "Corn", 5: "Soybeans", 36: "Alfalfa", 37: "Other Hay",
    61: "Fallow/Idle", 111: "Open Water", 121: "Developed",
    141: "Deciduous Forest", 176: "Grass/Pasture", 190: "Woody Wetlands"
}

# Map CDL classes to compact integer codes for the artifact
class_map = {v: i+1 for i, v in enumerate(cdl_legend.keys())}  # cdl_class -> compact_id
id_to_name = {i+1: name for i, (k, name) in enumerate(cdl_legend.items())}

output = {"years": years, "legend": id_to_name, "grids": {}}

for year in years:
    with rasterio.open(cdl_dir / f"CDL_{year}_19_1km.tif") as src:
        data = src.read(1)
        nrows, ncols = data.shape

    # Remap to compact codes, 0 = nodata/other
    compact = np.zeros_like(data, dtype=np.uint8)
    for cdl_cls, compact_id in class_map.items():
        compact[data == cdl_cls] = compact_id



    output["grids"][str(year)] = compact.tolist()
    print(f"{year}: {compact.shape}")

with open("iowa_cdl_grid.json", "w") as f:
    json.dump(output, f)
print("Saved iowa_cdl_grid.json")